# Métricas geométricas v2 — Buffer 600m + jerarquía viaria
**TFM – Álvaro Fernández-Uribarri Poveda · QuEA 2025–26**

## Dos mejoras respecto a v1, motivadas por Chen et al. (2021, CEUS)

### Mejora 1 — Ventana de cálculo ampliada (buffer 600m)

**Problema en v1**: las métricas se calculaban *dentro* de cada hexágono H3 res.9  
(diámetro ≈ 350m, área ≈ 0.1 km²). Un hexágono típico contiene 10–15 aristas viarias.  
Con tan pocas muestras, ningún estimador de distribución angular (R4, ADI) es estable.

**Solución**: calcular cada métrica usando todas las aristas dentro de un **buffer circular  
de 600m de radio** centrado en el hexágono, no sólo las aristas que caen dentro del polígono.

```
ANTES │ hex X → ~12 aristas dentro del hexágono → R4 muy ruidoso
AHORA │ hex X → ~100 aristas dentro del buffer 600m → R4 estable
```

El hexágono **no cambia**: sigue siendo la unidad de análisis y de predicción.  
Solo ampliamos la ventana de observación. Cada hexágono se solapa con los vecinos —  
eso es intencionado: captura el carácter morfológico local con suficiente contexto.

Chen et al. usan celdas de 1km² con imágenes de 2×2km (4× la unidad).  
Nuestro buffer de 600m sobre hexágonos de 350m es una ratio similar.

### Mejora 2 — Filtrado por jerarquía viaria

El Ensanche tiene callejones residenciales irregulares *entre* manzanas y avenidas  
perfectamente ortogonales. Si mezclamos todo, la señal ortogonal de las avenidas queda  
diluida. El casco medieval tiene callejuelas en todos los ángulos, pero también la  
Calle Mayor que va recta.

**Solución**: calcular métricas por separado sobre:
- **`_all`**: toda la red (como en v1, para comparación)
- **`_major`**: sólo calles `primary`, `secondary`, `tertiary` (las que definen el patrón morfológico)

Las calles `residential`, `living_street`, `pedestrian`, `footway` se excluyen de `_major`.

El diagnóstico Cohen d dirá cuál de las dos versiones discrimina mejor.

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import osmnx as ox
import warnings
from pathlib import Path
from shapely.ops import polygonize, unary_union

warnings.filterwarnings('ignore')
print('Imports OK')

In [ ]:
GPKG_IN = Path(r'C:\Users\Alvar\OneDrive\Desktop\estudios\economía\Artículos\MQUEA\madrid_morfologia_h3_v10.gpkg')
CRS     = 'EPSG:25830'
BUFFER_M = 600   # radio del buffer en metros

gdf = gpd.read_file(GPKG_IN).to_crs(CRS).reset_index(drop=True)
gdf['hex_idx'] = gdf.index

print(f'Hexágonos: {len(gdf):,}  |  CRS: {gdf.crs}')
if 'barrio_label' in gdf.columns:
    esp  = gdf['barrio_label'] == 0
    plan = gdf['barrio_label'] == 1
    print(f'Ground truth  →  Espontáneo: {esp.sum():,}  |  Planificado: {plan.sum():,}')

In [ ]:
# ── Red viaria: descargar con atributo highway ─────────────────────────────
print('Descargando red viaria (network_type=all)...')
G = ox.graph_from_place('Madrid, Spain', network_type='all', simplify=True)

# Bearings ANTES de proyectar (requiere CRS geográfico)
G = ox.add_edge_bearings(G)

# Proyectar DESPUÉS
G_proj = ox.project_graph(G, to_crs=CRS)

nodes_gdf, edges_gdf = ox.graph_to_gdfs(G_proj)
nodes_gdf = nodes_gdf.reset_index()[['osmid', 'geometry']]
edges_gdf = edges_gdf.reset_index()[['u', 'v', 'key', 'highway', 'bearing', 'length', 'geometry']]

print(f'Nodos: {len(nodes_gdf):,}  |  Aristas: {len(edges_gdf):,}')

# Normalizar columna 'highway' (puede ser str o list en OSMnx)
def hw_normalize(val):
    if isinstance(val, list):
        return val[0]
    return str(val)

edges_gdf['hw'] = edges_gdf['highway'].apply(hw_normalize)

MAJOR = {
    'motorway', 'motorway_link',
    'trunk',    'trunk_link',
    'primary',  'primary_link',
    'secondary','secondary_link',
    'tertiary', 'tertiary_link',
}

edges_major = edges_gdf[edges_gdf['hw'].isin(MAJOR)].copy()
edges_all   = edges_gdf.copy()

print(f'\nAristas totales:  {len(edges_all):,}')
print(f'Aristas major:    {len(edges_major):,}  ({len(edges_major)/len(edges_all)*100:.1f}%)')
print('\nDistribución de clases major:')
print(edges_major['hw'].value_counts().to_string())

In [ ]:
# ── Crear buffers de 600m alrededor de cada hexágono ─────────────────────
# Los centroides proyectados en EPSG:25830 permiten buffer en metros exactos.

gdf_buf = gdf[['hex_idx', 'geometry']].copy()
gdf_buf['geometry'] = gdf.geometry.centroid.buffer(BUFFER_M)

print(f'Buffers creados: {len(gdf_buf):,} × {BUFFER_M}m')
print(f'Área buffer: {gdf_buf.geometry.area.mean()/1e6:.3f} km²  '
      f'(vs. hexágono: {gdf.geometry.area.mean()/1e6:.3f} km²)')
print(f'Ratio contexto/unidad: {gdf_buf.geometry.area.mean()/gdf.geometry.area.mean():.1f}×')

In [ ]:
# ── Spatial join: aristas → buffers ──────────────────────────────────────
# Cada arista puede aparecer en VARIOS hexágonos (los cuyos buffers se solapan).
# Eso es correcto: buscamos una estimación suavizada del carácter local.

def join_edges_to_buffers(edges_df, label=''):
    """Asigna aristas a hexágonos por centroide dentro del buffer."""
    ecent = edges_df.copy()
    ecent['geometry'] = edges_df.geometry.centroid
    joined = gpd.sjoin(
        ecent[['u', 'v', 'bearing', 'length', 'geometry']],
        gdf_buf, how='inner', predicate='within'
    )
    joined['hex_idx'] = joined['hex_idx'].astype(int)
    print(f'  {label:<10} → {len(joined):,} asignaciones  '
          f'(media {len(joined)/len(gdf):.0f} aristas/hexágono)')
    return joined

print('Spatial join aristas → buffers:')
edges_buf_all   = join_edges_to_buffers(edges_all,   'all')
edges_buf_major = join_edges_to_buffers(edges_major, 'major')

# ── Nodos dentro de cada buffer ───────────────────────────────────────────
nodes_buf = gpd.sjoin(
    nodes_gdf[['osmid', 'geometry']],
    gdf_buf, how='inner', predicate='within'
)
nodes_buf['hex_idx'] = nodes_buf['hex_idx'].astype(int)
print(f'  Nodos     → {len(nodes_buf):,} asignaciones  '
      f'(media {len(nodes_buf)/len(gdf):.0f} nodos/hexágono)')

In [ ]:
# ── Funciones de cálculo ──────────────────────────────────────────────────

def circular_R(bearings_deg, k):
    """
    k-ésimo armónico circular. R_k = 1 → simetría perfecta de orden k.
    Invariante a rotación: R4=1 para cualquier cuadrícula sin importar su orientación.
    """
    theta = np.radians(np.asarray(bearings_deg, dtype=float))
    return float(np.sqrt(np.mean(np.cos(k * theta))**2 + np.mean(np.sin(k * theta))**2))


def bearing_alignment_score(bearings_deg, tol_deg=15.0):
    """
    BAS: % de calles alineadas con las 2 direcciones perpendiculares dominantes.
    Maneja cuadrículas de cualquier orientación.
    """
    b = np.asarray(bearings_deg, dtype=float) % 180
    hist, bin_edges = np.histogram(b, bins=36, range=(0, 180))
    centers  = (bin_edges[:-1] + bin_edges[1:]) / 2
    dominant = centers[np.argmax(hist)]
    perp     = (dominant + 90) % 180
    d_dom    = np.abs(((b - dominant + 90) % 180) - 90)
    d_perp   = np.abs(((b - perp     + 90) % 180) - 90)
    return float((np.minimum(d_dom, d_perp) < tol_deg).mean())


def adi_node(bearings_list):
    """ADI para un nodo. Sólo cruces con ≥ 3 calles."""
    b = sorted(float(x) % 360 for x in bearings_list)
    k = len(b)
    if k < 3:
        return np.nan
    diffs = np.diff(b + [b[0] + 360.0])
    return float(np.mean(np.abs(diffs - 360.0 / k)))


def compute_bearing_metrics(edges_joined, suffix=''):
    """
    Calcula R4, R6, R2, BAS y edge_len_cv por hexágono.
    suffix: '_all' o '_major' para nombrar las columnas.
    """
    def per_hex(grp):
        b = grp['bearing'].dropna().values
        l = grp['length'].dropna().values
        if len(b) < 6:   # umbral mínimo para estimación estable
            return pd.Series({f'R4{suffix}': np.nan, f'R6{suffix}': np.nan,
                              f'BAS{suffix}': np.nan, f'edge_len_cv{suffix}': np.nan,
                              f'n_edges{suffix}': len(b)})
        return pd.Series({
            f'R4{suffix}':          circular_R(b, 4),
            f'R6{suffix}':          circular_R(b, 6),
            f'BAS{suffix}':         bearing_alignment_score(b),
            f'edge_len_cv{suffix}': float(l.std() / l.mean()) if l.mean() > 0 else np.nan,
            f'n_edges{suffix}':     len(b),
        })
    return edges_joined.groupby('hex_idx').apply(per_hex)


def compute_adi(edges_joined, suffix=''):
    """
    Calcula ADI_mean por hexágono desde las aristas del buffer (vectorizado).
    """
    ev = edges_joined[edges_joined['bearing'].notna()][['u', 'v', 'bearing', 'hex_idx']].copy()
    ev['bearing'] = ev['bearing'].astype(float)

    # Construir bearings salientes de cada nodo dentro del contexto del buffer
    from_u = ev[['u', 'bearing', 'hex_idx']].rename(columns={'u': 'osmid'})
    from_v = ev[['v', 'bearing', 'hex_idx']].copy()
    from_v['bearing'] = (from_v['bearing'] + 180) % 360
    from_v = from_v.rename(columns={'v': 'osmid'})

    all_nb = pd.concat([from_u, from_v], ignore_index=True)

    # ADI por (nodo, hexágono): en el buffer de cada hex, ¿cómo de regulares son sus cruces?
    node_hex_bearings = all_nb.groupby(['hex_idx', 'osmid'])['bearing'].apply(list)
    adi_vals = node_hex_bearings.map(adi_node)

    adi_df = adi_vals.reset_index()
    adi_df.columns = ['hex_idx', 'osmid', 'ADI']

    result = adi_df.groupby('hex_idx')['ADI'].agg(
        **{f'ADI_mean{suffix}': 'mean',
           f'ADI_median{suffix}': 'median'}
    )
    return result

print('Funciones definidas OK')

In [ ]:
# ── Calcular todas las variantes ──────────────────────────────────────────
#
# Cuatro combinaciones:
#   all   × buffer600 → estimador robusto, todas las calles
#   major × buffer600 → estimador robusto, sólo calles estructurales

print('Calculando métricas de bearing (buffer 600m)...')
metrics_all   = compute_bearing_metrics(edges_buf_all,   suffix='_all')
metrics_major = compute_bearing_metrics(edges_buf_major, suffix='_major')
print('  R4, BAS OK')

print('Calculando ADI (buffer 600m)...')
adi_all   = compute_adi(edges_buf_all,   suffix='_all')
adi_major = compute_adi(edges_buf_major, suffix='_major')
print('  ADI OK')

# ── Resumen de cobertura ──────────────────────────────────────────────────
print(f'\n{"Métrica":<22}  {"Cobert.":>9}  {"Media":>7}  {"Std":>7}')
print('-' * 50)
for df, cols in [
    (metrics_all,   ['R4_all',   'BAS_all',   'edge_len_cv_all']),
    (metrics_major, ['R4_major', 'BAS_major', 'edge_len_cv_major']),
]:
    for c in cols:
        if c not in df.columns: continue
        s = df[c].dropna()
        print(f'{c:<22}  {len(s):>5,}({len(s)/len(gdf)*100:3.0f}%)  {s.mean():>7.3f}  {s.std():>7.3f}')
for df, cols in [
    (adi_all,   ['ADI_mean_all']),
    (adi_major, ['ADI_mean_major']),
]:
    for c in cols:
        if c not in df.columns: continue
        s = df[c].dropna()
        print(f'{c:<22}  {len(s):>5,}({len(s)/len(gdf)*100:3.0f}%)  {s.mean():>7.2f}°  {s.std():>7.2f}°')

In [ ]:
# ── Manzanas: polygonize global (no cambia respecto a v1) ─────────────────
# Las manzanas se calculan con la red completa; el buffer no aplica aquí
# porque la manzana es una entidad global.

print('Polygonizando red viaria completa (2-4 min)...')
merged    = unary_union(edges_gdf['geometry'].tolist())
all_blocks = list(polygonize(merged))
print(f'Bloques crudos: {len(all_blocks):,}')

blocks_gdf = gpd.GeoDataFrame({'geometry': all_blocks}, crs=CRS)
blocks_gdf['area']        = blocks_gdf.geometry.area
blocks_gdf['perimeter']   = blocks_gdf.geometry.length
blocks_gdf['compactness'] = 4 * np.pi * blocks_gdf['area'] / (blocks_gdf['perimeter'] ** 2)

blocks_gdf = blocks_gdf[(blocks_gdf['area'] > 500) & (blocks_gdf['area'] < 200_000)].copy()
print(f'Manzanas filtradas (500–200k m²): {len(blocks_gdf):,}')

# Asignar a hexágono por centroide (dentro del polígono del hex, NO del buffer)
# Cada manzana pertenece a UN solo hexágono
blocks_cent = blocks_gdf.copy()
blocks_cent['geometry'] = blocks_gdf.geometry.centroid

hex_polys = gdf[['hex_idx', 'geometry']]
blocks_hex = gpd.sjoin(
    blocks_cent[['area', 'compactness', 'geometry']],
    hex_polys, how='left', predicate='within'
).dropna(subset=['hex_idx'])
blocks_hex['hex_idx'] = blocks_hex['hex_idx'].astype(int)

def block_stats(grp):
    if len(grp) < 2:
        return pd.Series({'block_area_cv': np.nan, 'block_compact_mean': np.nan, 'n_blocks': len(grp)})
    a = grp['area'].values
    return pd.Series({
        'block_area_cv':      float(a.std() / a.mean()) if a.mean() > 0 else np.nan,
        'block_compact_mean': float(grp['compactness'].mean()),
        'n_blocks':           len(grp),
    })

block_metrics = blocks_hex.groupby('hex_idx').apply(block_stats)
print(f'Hexágonos con métricas de manzana: {block_metrics["block_area_cv"].notna().sum():,}')

In [ ]:
# ── Unir todo al GeoDataFrame principal ──────────────────────────────────

result = (
    gdf
    .join(metrics_all  [['R4_all',   'BAS_all',   'edge_len_cv_all',   'n_edges_all']],   on='hex_idx', how='left')
    .join(metrics_major[['R4_major', 'BAS_major', 'edge_len_cv_major', 'n_edges_major']], on='hex_idx', how='left')
    .join(adi_all  [['ADI_mean_all',   'ADI_median_all']],   on='hex_idx', how='left')
    .join(adi_major[['ADI_mean_major', 'ADI_median_major']], on='hex_idx', how='left')
    .join(block_metrics[['block_area_cv', 'block_compact_mean', 'n_blocks']], on='hex_idx', how='left')
)

# Métrica combinada: grid_crystallinity (max de R4 en ambas versiones)
result['grid_crystallinity_all']   = result['R4_all']
result['grid_crystallinity_major'] = result['R4_major']

NUEVAS = [
    'R4_all', 'R4_major',
    'BAS_all', 'BAS_major',
    'ADI_mean_all', 'ADI_mean_major',
    'edge_len_cv_all', 'edge_len_cv_major',
    'block_area_cv', 'block_compact_mean',
]

print('GeoDataFrame ensamblado OK')
print(f'Columnas nuevas: {len(NUEVAS)}')

In [ ]:
# ── Cohen d: diagnóstico comparativo ─────────────────────────────────────
#
# Comparamos:
#   bearing_entropy (v4 original, DENTRO del hex)
#   R4_all         (buffer 600m, todas las calles)
#   R4_major       (buffer 600m, calles estructurales)
#   ...y el resto de variantes nuevas

if 'barrio_label' not in result.columns:
    print('barrio_label no encontrado.')
else:
    esp  = result['barrio_label'] == 0
    plan = result['barrio_label'] == 1

    VARS_DIAG = ['bearing_entropy'] + NUEVAS  # bearing_entropy como referencia

    print(f'Cohen d  —  Espontáneo: {esp.sum()}  |  Planificado: {plan.sum()}')
    print()
    print(f'{"Variable":<25}  {"Esp":>8}  {"Plan":>8}  {"Cohen d":>9}  Veredicto')
    print('─' * 82)

    resultados_d = {}
    for v in VARS_DIAG:
        if v not in result.columns:
            print(f'{v:<25}  NO EXISTE'); continue
        col = pd.to_numeric(result[v], errors='coerce')
        me  = col[esp].mean()
        mp  = col[plan].mean()
        sp  = col[esp | plan].std()
        d   = (me - mp) / sp if sp > 0 else np.nan
        resultados_d[v] = d

        if   pd.isna(d):  icon = '—'
        elif abs(d) < 0.2: icon = '❌ ruido'
        elif abs(d) < 0.5: icon = '⚠️ marginal'
        elif d > 0.5:      icon = '✅ → espontáneo'
        else:              icon = '✅ → planificado'

        # Indicar si es una mejora sobre bearing_entropy
        ref_d = resultados_d.get('bearing_entropy', 0)
        mejora = ''
        if v != 'bearing_entropy' and not pd.isna(d):
            if abs(d) > abs(ref_d) + 0.1:
                mejora = ' ↑ MEJOR que bearing_entropy'

        print(f'{v:<25}  {me:>8.3f}  {mp:>8.3f}  {d:>+9.3f}  {icon}{mejora}')

    print()
    print('─' * 82)
    print('Ranking por |Cohen d|:')
    ranking = sorted(resultados_d.items(), key=lambda x: abs(x[1]) if not pd.isna(x[1]) else 0, reverse=True)
    for i, (v, d) in enumerate(ranking[:8], 1):
        print(f'  {i}. {v:<28} d = {d:+.3f}')

In [ ]:
# ── Mapas: comparación all vs major para R4 y ADI ─────────────────────────

Path('imagenes').mkdir(exist_ok=True)

try:
    barrios = ox.features_from_place('Madrid, Spain', tags={'admin_level': '10'})
    barrios = barrios[barrios.geometry.type.isin(['Polygon','MultiPolygon'])][['name','geometry']]
    barrios = barrios.to_crs(CRS).reset_index(drop=True)
    tiene_barrios = True
except Exception:
    tiene_barrios = False

PLOT_CFG = [
    ('R4_all',         'RdBu', (0, 1),    'R4 — todas las calles (buffer 600m)\nazul=cuadrícula, rojo=orgánico'),
    ('R4_major',       'RdBu', (0, 1),    'R4 — sólo calles primary/secondary/tertiary\n(señal morfológica estructural)'),
    ('BAS_major',      'RdBu', (0, 1),    'BAS — alineación con ejes dominantes\n(sólo calles estructurales)'),
    ('ADI_mean_all',   'RdBu_r', None,    'ADI — desviación ángulos en cruces, todas\n(alto=irregular=espontáneo)'),
    ('ADI_mean_major', 'RdBu_r', None,    'ADI — desviación ángulos en cruces, major\n(más diagnóstico del patrón estructural)'),
    ('block_area_cv',  'RdBu_r', None,    'CV área de manzanas\n(alto=desigual=espontáneo)'),
]

fig, axes = plt.subplots(2, 3, figsize=(30, 20))
axes = axes.flatten()

for ax, (col, cmap, vlims, title) in zip(axes, PLOT_CFG):
    vals = pd.to_numeric(result[col], errors='coerce')
    if vlims:
        vmin, vmax = vlims
    else:
        vabs = float(np.nanquantile(vals.abs(), 0.97))
        vmin, vmax = -vabs, vabs

    result.plot(
        column=col, ax=ax, cmap=cmap, vmin=vmin, vmax=vmax,
        linewidth=0.1, edgecolor='white', legend=True,
        legend_kwds={'shrink': 0.55, 'orientation': 'horizontal', 'pad': 0.02},
        missing_kwds={'color': '#DDDDDD'},
    )
    if tiene_barrios:
        barrios.boundary.plot(ax=ax, color='black', linewidth=0.4, alpha=0.4)

    # Cohen d en el título si disponible
    d_val = resultados_d.get(col, np.nan) if 'resultados_d' in dir() else np.nan
    d_str = f'  |d|={abs(d_val):.3f}' if not pd.isna(d_val) else ''
    ax.set_title(f'{title}\n(n={int(vals.notna().sum()):,}{d_str})', fontsize=9.5)
    ax.set_axis_off()

plt.suptitle(
    f'Métricas geométricas v2 — Buffer {BUFFER_M}m + jerarquía viaria — H3 res.9 Madrid',
    fontsize=13, y=1.01
)
plt.tight_layout()
plt.savefig('imagenes/metricas_geometricas_v2.png', dpi=200, bbox_inches='tight')
plt.show()
print('Guardado: imagenes/metricas_geometricas_v2.png')

In [ ]:
# ── Comparación directa: bearing_entropy vs. R4_major ─────────────────────

fig, axes = plt.subplots(1, 3, figsize=(33, 11))

cfgs = [
    ('bearing_entropy', 'RdBu_r', None,   'bearing_entropy (v4, dentro del hex)\nCohen d ≈ +0.09  ❌'),
    ('R4_all',          'RdBu',   (0, 1), f'R4_all (buffer {BUFFER_M}m, todas las calles)'),
    ('R4_major',        'RdBu',   (0, 1), f'R4_major (buffer {BUFFER_M}m, primary/sec/ter)'),
]

for ax, (col, cmap, vlims, title) in zip(axes, cfgs):
    vals = pd.to_numeric(result[col], errors='coerce') if col in result.columns else pd.Series(dtype=float)
    if vlims:
        vmin, vmax = vlims
    else:
        vabs = float(np.nanquantile(vals.abs().dropna(), 0.97))
        vmin, vmax = -vabs, vabs

    result.plot(column=col, ax=ax, cmap=cmap, vmin=vmin, vmax=vmax,
                linewidth=0.1, edgecolor='white', legend=True,
                legend_kwds={'shrink': 0.55, 'orientation': 'horizontal', 'pad': 0.02},
                missing_kwds={'color': '#DDDDDD'})
    if tiene_barrios:
        barrios.boundary.plot(ax=ax, color='black', linewidth=0.4, alpha=0.4)

    d_val = resultados_d.get(col, np.nan) if 'resultados_d' in dir() else np.nan
    d_str = f'\nCohen d = {d_val:+.3f}' if not pd.isna(d_val) else ''
    ax.set_title(f'{title}{d_str}', fontsize=11)
    ax.set_axis_off()

plt.suptitle('Evolución del discriminador de cuadrícula: entropía → armónico circular',
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig('imagenes/comparacion_entropy_vs_R4_v2.png', dpi=200, bbox_inches='tight')
plt.show()
print('Guardado: imagenes/comparacion_entropy_vs_R4_v2.png')

In [ ]:
# ── Análisis de sensibilidad: ¿importa el tamaño del buffer? ─────────────
# Probamos con 300m, 600m y 900m para ver cómo varía Cohen d de R4_major.
# Ejecutar sólo si quieres investigar el parámetro óptimo.

if 'barrio_label' in result.columns:
    esp  = result['barrio_label'] == 0
    plan = result['barrio_label'] == 1

    print('Sensibilidad de R4_major al tamaño del buffer:')
    print(f'{"Buffer (m)":<12}  {"n_edges/hex":>12}  {"Cohen d":>9}')
    print('-' * 38)

    for buf_m in [300, 400, 500, 600, 750, 900]:
        # Buffer temporal
        gdf_buf_tmp = gdf[['hex_idx', 'geometry']].copy()
        gdf_buf_tmp['geometry'] = gdf.geometry.centroid.buffer(buf_m)

        ecent = edges_major.copy()
        ecent['geometry'] = edges_major.geometry.centroid
        joined_tmp = gpd.sjoin(
            ecent[['u', 'v', 'bearing', 'length', 'geometry']],
            gdf_buf_tmp, how='inner', predicate='within'
        )
        joined_tmp['hex_idx'] = joined_tmp['hex_idx'].astype(int)

        met_tmp = compute_bearing_metrics(joined_tmp, suffix='_tmp')
        n_mean  = joined_tmp.groupby('hex_idx').size().mean()

        col = result.join(met_tmp[['R4_tmp']], on='hex_idx', how='left')['R4_tmp']
        col = pd.to_numeric(col, errors='coerce')
        me, mp = col[esp].mean(), col[plan].mean()
        sp = col[esp | plan].std()
        d  = (me - mp) / sp if sp > 0 else np.nan

        print(f'{buf_m:<12}  {n_mean:>12.0f}  {d:>+9.3f}')

    print('\n  → El buffer óptimo es el que maximiza |d| sin ser tan grande que')
    print('    pierda resolución espacial (varios barrios distintos en el buffer).')

In [ ]:
# ── Exportar GeoPackage ───────────────────────────────────────────────────

OUT = Path(r'C:\Users\Alvar\OneDrive\Desktop\estudios\economía\Artículos\MQUEA\madrid_morfologia_h3_metricas_geometricas_v2.gpkg')

cols_exportar = [c for c in result.columns if c != 'hex_idx']
result[cols_exportar].to_file(OUT, driver='GPKG')
print(f'Exportado: {OUT.name}')
print(f'Columnas nuevas: {NUEVAS}')